# Logistic Regression — Heart Disease PredictionThis notebook builds a logistic regression classifier for the **UCI Cleveland Heart Disease** dataset. Each row represents a patient; the target is whether the patient has heart disease (binary).**Goals:**1. Walk through a complete classification workflow (load, EDA, scale, fit, evaluate)2. Evaluate using **accuracy**, **precision/recall**, **confusion matrix**, and **ROC-AUC**3. Interpret the learned coefficients**Dataset:** 303 patients, 13 features (age, sex, chest pain type, blood pressure, cholesterol, etc.). Source: [UCI](https://archive.ics.uci.edu/dataset/45/heart+disease).

In [ ]:
import osimport urllib.requestDATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data") if os.path.basename(os.getcwd()) == "notebooks" else "data"os.makedirs(DATA_DIR, exist_ok=True)def download_if_needed(url, filename):    """Download a CSV if it doesn't already exist locally."""    path = os.path.join(DATA_DIR, filename)    if not os.path.exists(path):        print(f"Downloading {filename} from {url}")        urllib.request.urlretrieve(url, path)    return pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import (    accuracy_score, classification_report,    confusion_matrix, roc_curve, auc,)sns.set_style("whitegrid")np.random.seed(42)

## 1. Load the data

In [ ]:
path = download_if_needed(    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/processed.cleveland.data.csv",    "heart_disease.csv",)cols = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",        "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"]df = pd.read_csv(path, header=None, names=cols, na_values="?").dropna()# Original target: 0=healthy, 1-4=disease severity. Binarize:df["target"] = (df["target"] > 0).astype(int)print("Shape:", df.shape)print("Class balance:", df["target"].value_counts().to_dict())df.head()

## 2. Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))df["age"].hist(bins=20, ax=axes[0])axes[0].set_title("Age distribution"); axes[0].set_xlabel("age")df.groupby("target")["thalach"].mean().plot(    kind="bar", ax=axes[1], color=["steelblue", "salmon"])axes[1].set_title("Avg max heart rate (thalach) by class")axes[1].set_xticklabels(["healthy", "disease"], rotation=0)plt.tight_layout(); plt.show()

## 3. Split + scale

In [ ]:
X = df.drop(columns="target").valuesy = df["target"].valuesX_tr, X_te, y_tr, y_te = train_test_split(    X, y, test_size=0.25, random_state=42, stratify=y)scaler = StandardScaler()X_tr_s = scaler.fit_transform(X_tr)X_te_s = scaler.transform(X_te)

## 4. Fit logistic regression

In [ ]:
clf = LogisticRegression(max_iter=2000, random_state=42).fit(X_tr_s, y_tr)preds = clf.predict(X_te_s)probs = clf.predict_proba(X_te_s)[:, 1]print(f"Accuracy: {accuracy_score(y_te, preds):.4f}\n")print(classification_report(y_te, preds, target_names=["healthy", "disease"]))

## 5. Confusion matrix

In [ ]:
cm = confusion_matrix(y_te, preds)fig, ax = plt.subplots(figsize=(5, 4))sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",            xticklabels=["healthy", "disease"],            yticklabels=["healthy", "disease"], ax=ax)ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")ax.set_title("Confusion Matrix"); plt.tight_layout(); plt.show()

## 6. ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_te, probs)roc_auc = auc(fpr, tpr)plt.figure(figsize=(6, 5))plt.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}")plt.plot([0, 1], [0, 1], "k--", lw=1)plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")plt.title("ROC Curve"); plt.legend(); plt.tight_layout(); plt.show()

## 7. Coefficient interpretation

In [ ]:
coef_df = pd.DataFrame({    "feature": df.drop(columns="target").columns,    "coef": clf.coef_[0],}).assign(abs_coef=lambda d: d.coef.abs()).sort_values("abs_coef", ascending=False)coef_df.head(10).plot.barh(x="feature", y="coef", figsize=(8, 5),                           legend=False, color="steelblue")plt.gca().invert_yaxis()plt.title("Top 10 features by |coefficient| (positive ↑ disease risk)")plt.tight_layout(); plt.show()coef_df.round(3)

## Takeaways- Logistic regression hits ~85% accuracy with AUC ≈ 0.92 on this dataset — very respectable for a simple linear model.- The `thal` (thallium stress test result), `cp` (chest pain type), and `ca` (number of major vessels) features dominate.- Higher max heart rate (`thalach`) is *protective* — its coefficient is negative.- Feature scaling matters because logistic regression's default L2 regularizer is sensitive to scale.